# RiNALMo

In [ ]:
import os
import sys
import pandas as pd
import subprocess
import shutil
from pathlib import Path

In [ ]:
# Setup paths
base = Path.cwd()
tools_path = base.parent / 'tools'
rinalmo_path = tools_path / 'RiNALMo'
weights_path = rinalmo_path / 'weights' / 'rinalmo_giga_ss_bprna_ft.pt'
virus_data_path = base.parent / 'data' / 'viruses.fasta'

# Clone RiNALMo if not exists
if not rinalmo_path.exists():
    print(f"Cloning RiNALMo to {rinalmo_path}...")
    os.system(f"cd {tools_path} && git clone https://github.com/lbcb-sci/RiNALMo.git")
else:
    print(f"RiNALMo already exists at {rinalmo_path}")

# Download weights if not exists
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    print(f"Downloading weights to {weights_path}...")
    os.system(f"cd {weights_path.parent} && wget https://zenodo.org/records/15043668/files/rinalmo_micro_pretrained.pt")
else:
    print(f"Weights already exists at {weights_path}")

# Data Loading
Load virus sequences for prediction

In [ ]:
def read_virus_fasta(path: str):
    """Read virus sequences from FASTA format"""
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    if len(lines) % 3 != 0:
        raise ValueError(f"FASTA file has {len(lines)} lines, not divisible by 3")
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

def dot_bracket_to_ct(sequence, structure, seq_name):
    """Convert dot-bracket structure to CT format"""
    seq_len = len(sequence)
    if len(structure) != seq_len:
        raise ValueError(f"Sequence length ({seq_len}) != structure length ({len(structure)})")
    
    pairs = {}
    stack = []
    
    for i, char in enumerate(structure):
        if char == '(':
            stack.append(i)
        elif char == ')':
            if stack:
                j = stack.pop()
                pairs[j+1] = i+1
                pairs[i+1] = j+1
    
    ct_lines = [f"{seq_len} {seq_name}"]
    
    for i in range(1, seq_len + 1):
        nucleotide = sequence[i-1]
        prev_pos = i - 1 if i > 1 else 0
        next_pos = i + 1 if i < seq_len else 0
        pair_pos = pairs.get(i, 0)
        
        ct_lines.append(f"{i} {nucleotide} {prev_pos} {next_pos} {pair_pos} {i}")
    
    return '\n'.join(ct_lines)

viruses = read_virus_fasta(str(virus_data_path))
print(f'Loaded {len(viruses)} virus sequences')

# Prediction
Process each sequence individually and predict structures

In [ ]:
import time

def run_prediction(virus_id):
    """Run RiNALMo prediction for single sequence"""
    start_time = time.time()
    temp_data_dir = base / 'temp_rinalmo_data'
    if temp_data_dir.exists():
        shutil.rmtree(temp_data_dir)
    
    # Use bpRNA dataset structure as expected by RiNALMo
    bprna_dir = temp_data_dir / 'bpRNA'
    test_dir = bprna_dir / 'test'
    test_dir.mkdir(parents=True, exist_ok=True)
    
    seq = viruses.loc[virus_id]['sequence']
    struct = viruses.loc[virus_id]['structure']
    ct_content = dot_bracket_to_ct(seq, struct, virus_id)
    ct_path = test_dir / f"{virus_id}.ct"
    ct_path.write_text(ct_content)
    
    output_dir = base / 'temp_rinalmo_output'
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Set environment variable to use GPU 1 specifically
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '0'
    
    cmd = [
        'python', str(rinalmo_path / 'train_sec_struct_prediction.py'),
        str(temp_data_dir),
        '--test_only',
        '--init_params', str(weights_path),
        '--dataset', 'bpRNA',
        '--output_dir', str(output_dir),
        '--accelerator', 'gpu',
        '--devices', '1'
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    
    if result.returncode != 0:
        return None, None, None
    
    ct_files = list(output_dir.glob('*.ct'))
    if ct_files:
        ct_file = ct_files[0]
        # Convert CT to dot-bracket using ct2dot.py script
        dot_file = ct_file.parent / f"{ct_file.stem}.dot"
        os.system(f"python ct2dot.py {str(ct_file)} {str(dot_file)} -f full -q")
        
        # Read converted dot file
        if dot_file.exists():
            with open(dot_file, 'r') as f:
                lines = f.readlines()
            
            sequence = ''
            structure = ''
            for line in lines:
                if line.startswith('>'):
                    continue
                elif line and line[0] in 'ACGU':
                    sequence += line.strip()
                else:
                    structure += line.strip()
            
            elapsed_time = time.time() - start_time
            return sequence, structure, elapsed_time
    
    return None, None, None

output_dir = base.parent / 'prediction'
output_dir.mkdir(exist_ok=True)
output_fasta = output_dir / 'RiNALMo.fasta'

if output_fasta.exists():
    output_fasta.unlink()


In [ ]:
virus_ids = list(viruses.index)

print(f"{'Idx':>3}\t{'Virus':<20}\t{'Length':<5}\t{'Status'}")
print("-" * 60)

for i, vid in enumerate(virus_ids, 1):
    seq_len = len(viruses.loc[vid]['sequence'])
    print(f"{i:3d}/{len(virus_ids)}\t{vid:<20}\t{seq_len:<5}\t", end='', flush=True)
    
    try:
        sequence, structure, elapsed_time = run_prediction(vid)
        if sequence and structure and elapsed_time is not None:
            with open(output_fasta, 'a') as f:
                f.write(f">{vid}\n")
                f.write(f"{sequence}\n")
                f.write(f"{structure}\n")
            print(f"{elapsed_time:.1f} s")
        else:
            print("Fail")
    except Exception as e:
        print(f"Fail")

print(f"Results saved to: {output_fasta}")
